In [ ]:
%matplotlib inline

import os, random, time, sys, warnings
from timeit import default_timer as timer
from datetime import timedelta
from datetime import datetime

import numpy as np
import scipy as sp
from scipy.spatial import distance
from scipy import signal
import pandas as pd
from tqdm import tqdm
import bct
from scipy.optimize import curve_fit

import neurogym as ngym
import torch
import torch.nn as nn

from src.neural_network import RNN, run_testing
from src.utils import normalize_x, map_kernel_to_epochs, build_reg_ken, compute_rlfp, get_weight_masks, get_weight_masks_schaefer, autocorr, get_file_str

# import plotting libraries
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10})
plt.rcParams["svg.fonttype"] = "none"
plt.rc('font', family='DejaVu Sans')
import seaborn as sns
sns.set_style("white")
from src.plotting import my_reg_plot

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')

In [ ]:
save_figs = True

In [ ]:
# directories
datadir = '/home/lindenmp/research_projects/neuro_rnn/data'
modeldir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/pytorch/model'
outdir = '/home/lindenmp/research_projects/neuro_rnn/results/figs'

# data parameters
task = 'PerceptualDecisionMaking-v0'
# task = 'MultiSensoryIntegration-v0'
# task = 'ContextDecisionMaking-v0'
dt = 100
batch_size = 32
decision = 300
if task == 'PerceptualDecisionMaking-v0':
    seq_len = 22
elif task == 'MultiSensoryIntegration-v0':
    seq_len = 11
elif task == 'ContextDecisionMaking-v0':
    seq_len = 13
seq_len = seq_len + int((decision - 100) / dt)
seq_len_multi = 5
seq_len = seq_len * seq_len_multi
print(seq_len)

# RNN model and training parameters
rnn_model = 'rnn-tanh'
hidden_size = 100
n_runs = 25
n_epochs = 30000
lr = 0.001

# regularization parameters
reg_type = 'l2'
reg_weight = 0.0015
mask_weights = True
# kernel_type = 'sa_axis'
kernel_type = 'euclidean'
# kernel_type = None

if mask_weights and hidden_size == 50:
    n_io = '9-13'
if mask_weights and hidden_size == 100:
    n_io = '14-27'
if mask_weights and hidden_size == 200:
    n_io = '31-52'

In [ ]:
timing = {'decision': decision}
env_kwargs = {'dt': dt, 'timing': timing}

config = {
    'datadir': datadir, 'outdir': outdir,
    'task': task, 'dt': dt, 'seq_len': seq_len, 'batch_size': batch_size,  # data parameters
    'rnn_model': rnn_model, 'hidden_size': hidden_size, 'n_runs': n_runs, 'n_epochs': n_epochs, 'lr': lr, 'mask_weights': mask_weights,  # RNN model and training parameters
    'reg_type': reg_type, 'reg_weight': reg_weight, 'kernel_type': kernel_type, # regularization parameters
    'env_kwargs': env_kwargs,
    'n_io': n_io
}

file_str = get_file_str(config)
print(file_str)

# Load activity data

In [ ]:
# load data
log_args = np.load(os.path.join(modeldir, file_str + '.npy'), allow_pickle=True).item()
activity = log_args['hidden_activity']
activity = activity[:, :100, :, :]

print(activity.shape)
n_timepoints = activity.shape[2]
n_trials = activity.shape[1]
trim = int(n_timepoints - (decision / 100))

In [ ]:
activity_zscore = np.zeros(activity.shape)
activity_zscore = activity_zscore[:, :, :trim, :]
print(activity_zscore.shape)

for run in tqdm(np.arange(n_runs)):
    for trial in np.arange(n_trials):
        activity_zscore[run, trial] = sp.stats.zscore(activity[run, trial, :trim], axis=0)

    activity_zscore[run] = sp.stats.zscore(activity_zscore[run], axis=0)

# RLFP

In [ ]:
dt_seconds = dt / 1000
num_bands = 5
sample_freq = 1 / dt_seconds
nyq_freq = sample_freq / 2
print(sample_freq, nyq_freq)

band_of_interest = 1
band_intervals = np.linspace(0, nyq_freq, num_bands + 1)
band_freq_range = band_intervals[band_of_interest - 1:band_of_interest + 1]

In [ ]:
activity_reshape = np.reshape(activity_zscore.copy(), (-1, activity_zscore.shape[-1]))
print(activity_reshape.shape)
rlfp = np.zeros(hidden_size)
for neuron in tqdm(np.arange(hidden_size)):
    rlfp[neuron] = compute_rlfp(ts=activity_reshape[:, neuron], tr=dt_seconds, low=None, high=None, num_bands=num_bands, band_of_interest=band_of_interest)

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(5, 5))
my_reg_plot(np.arange(hidden_size), rlfp, 'Neuron', 'RLFP', ax=ax, fontsize=12, annotate='both')
ax.set_title('{:}, {:}'.format(rnn_model, kernel_type))

if save_figs:
    f.savefig(os.path.join(outdir, '{0}_{1}.png'.format(file_str, 'rlfp')), dpi=300, bbox_inches='tight', pad_inches=0.01)

# Autocorrelation, tau

In [ ]:
# The exponential decay function
def exp_decay(x, tau, init):
    return init*np.e**(-x/tau)

In [ ]:
tau = np.zeros((n_runs, n_trials, hidden_size))

for run in tqdm(np.arange(n_runs)):
    for trial in np.arange(n_trials):
        for neuron in np.arange(hidden_size):
            try:
                t_ac, ac = autocorr(activity[run, trial, :, neuron], max_lag=trim)
                popt, pcov = curve_fit(exp_decay, t_ac, ac)
                fit_tau, fit_init = popt
                tau[run, trial, neuron] = fit_tau
            except:
                tau[run, trial, neuron] = np.nan

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(5, 5))
my_reg_plot(np.arange(hidden_size), np.nanmean(np.nanmean(tau, axis=0), axis=0), 'Neuron', 'Autocorrelation decay (tau)', ax=ax, fontsize=12, annotate='both')
ax.set_title('{:}, {:}'.format(rnn_model, kernel_type))

if save_figs:
    f.savefig(os.path.join(outdir, '{0}_{1}.png'.format(file_str, 'tau')), dpi=300, bbox_inches='tight', pad_inches=0.01)